# Pressure sensor calibration & cleaning, for OCEANLYZ

Two-part pipeline: fit a calibration from a rope-drop test, then apply it
to a deployment/wave file, trimmed to a survey window you specify. Output
is ready for [OCEANLYZ](https://github.com/akarimp/Oceanlyz).

**Part A - Calibration:** one continuous file where the sensor was held at
several known depths in turn (down, then back up). The notebook finds each
depth-hold automatically, corrects for barometric/zero drift using the
flat air readings before and after the test, and fits
`corrected_pa = slope * raw_pa + intercept`.

**Part B - Deployment:** applies that fit to your wave/deployment file,
trimmed to a survey start/end you provide (not auto-detected). If you have
a rough depth measured at recovery, it's checked against the modeled depth
at the end of the record as a sanity check only.

**Output:** results are saved in a folder named after the site, e.g.
`unive1/`. Inside it, `oceanlyz.csv` is a single column of calibrated
pressure (Pa), no header - the file to feed into OCEANLYZ
(`InputType='pressure'`). `clean.csv` has the full detail (raw + calibrated
values, timestamps). `staircase.png` and `calibration_fit.png` are for
checking the calibration; `qc.png` for checking the deployment trim.

This notebook does not run any wave analysis - only calibration and cleaning.

## Code layout

```
scripts/
├── parsing.py       - reads the raw logger CSV into a DataFrame
├── submersion.py    - auto-detect or manually set the in-water window
├── staircase.py     - finds depth-holds in a rope-drop calibration file
├── calibration.py   - fits/applies the pressure calibration
└── pipeline.py       - process_calibration() / process_deployment()
```

**If you replace the files in `scripts/` after already running this
notebook once in the current session**, restart the runtime first
(`Runtime -> Restart session`) before rerunning - Python keeps an
already-imported module in memory and won't notice files on disk changed
otherwise.

**Expected folder layout** (unzip the delivered package into a Drive
folder alongside your data):

```
your_project_folder/
├── pressure_calibrate_and_prepare.ipynb
├── scripts/
│   └── ... (parsing.py, submersion.py, staircase.py, calibration.py, pipeline.py)
└── Data/
    ├── calibration_file.csv
    └── wave_file.csv
```

## 1. Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Set the working directory

This is the folder you unzipped `scripts/` and your `Data/` folder into
(see the layout above). Type its path, or just press Enter to use the
current directory as-is.

In [ ]:
import os

PROJECT_DIR = input(f"Working directory [{os.getcwd()}]: ").strip() or os.getcwd()
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", sorted(os.listdir(".")))


## 3. Imports

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("."))  # so "scripts" (in the working directory) is importable
from scripts import (
    process_calibration, print_calibration_report,
    process_deployment, print_report,
    plot_staircase, plot_calibration_fit, plot_qc,
)


## 4. Configuration

### Calibration (`CALIB`) - the rope-drop test file

- `calib_csv` - path to the raw calibration file, relative to `INPUT_DIR`.
- `start_time` / `end_time` - bracket the whole test, including a bit of
  flat, in-air record on both ends (a few seconds is enough) - used to
  find the pre-/post-test air readings for the barometric correction, and
  to isolate the test from everything else in the file.
- `nominal_depths` - the rope depth (m) at each hold, in the order you did
  them, e.g. down 0.5/1/1.5/2 m then back up 1.5/1/0.5 m. Must match the
  number of holds the notebook actually finds in section 5 below - if it
  doesn't, check the printed table/plot and adjust.
- `rope_offset_m` - added to every value in `nominal_depths`: the distance
  from the rope's depth markings down to the sensor's own pressure port.
  `0.0` if the rope depths already refer to the sensor itself.
- `rho` - water density (kg/m³) used for the dip test: ~1025 seawater,
  ~1000 freshwater.

### Deployment (`SITE`) - the wave/deployment file

- `name` - used to name this run's output folder. Must be unique per site.
- `raw_csv` - path to the raw deployment file, relative to `INPUT_DIR`.
- `survey_start` / `survey_end` - the deployment window, entered directly
  (not auto-detected) - e.g. from field notes of when the sensor went in
  and came back out.
- `height_above_bed_m` - sensor height above the seabed (`0.0` if it sits
  directly on the bed). Not used here, but needed later by OCEANLYZ.
- `rho` - water density at the deployment site (can differ from the
  calibration test's `rho` above, e.g. bucket test in freshwater, real
  deployment in seawater).
- `recovery_depth_m` - optional. A rough depth measured right at recovery
  (not precise) - used only as a sanity cross-check against the modeled
  depth at the end of the record, not as a calibration input. Set to
  `None` to skip.

This cell also defines and creates `site_dir` - the one folder, named
after `SITE["name"]`, that every output from this run goes into. It's
created here, once, so every later cell reuses the same path.

In [ ]:
INPUT_DIR = "Data"   # folder holding your raw CSV files
OUTPUT_DIR = "."     # where per-site result folders get created

CALIB = {
    "calib_csv": "calibration_file.csv",
    "start_time": "2026-08-18 13:20:00",
    "end_time": "2026-08-18 13:25:00",
    "nominal_depths": [0.5, 1.0, 1.5, 2.0, 1.5, 1.0, 0.5],
    "rope_offset_m": 0.0,
    "rho": 1025,
}

SITE = {
    "name": "unive1",
    "raw_csv": "wave_file.csv",
    "survey_start": "2026-08-21 06:00:00",
    "survey_end": "2026-08-22 16:00:00",
    "height_above_bed_m": 0.0,
    "rho": 1025,
    "recovery_depth_m": None,
}

site_dir = os.path.join(OUTPUT_DIR, SITE["name"])
os.makedirs(site_dir, exist_ok=True)
print("This run's output folder:", site_dir)


## 5. Run the calibration

`process_calibration()` does, in order:

1. `load_raw()` on `CALIB["calib_csv"]`.
2. Trims to `[start_time, end_time]` and barometric/zero-drift-corrects
   using the flat stretches right before and after the test.
3. Finds each depth-hold in between (`detect_calibration_steps()`).
4. Labels them with `nominal_depths` (+ `rope_offset_m`), converts each
   known depth to a target pressure (using `rho`), and fits
   `slope`/`intercept` by least squares.

Check the printed table below: if the number of holds found doesn't match
`len(CALIB["nominal_depths"])`, look at the plot in the next cell and
adjust `start_time`/`end_time` or `CALIB["nominal_depths"]` before
trusting the fit.

In [ ]:
cal = process_calibration(
    CALIB["calib_csv"], CALIB["start_time"], CALIB["end_time"],
    CALIB["nominal_depths"], CALIB["rho"],
    input_dir=INPUT_DIR, rope_offset_m=CALIB["rope_offset_m"],
    expected_steps=len(CALIB["nominal_depths"]),
)
print_calibration_report(cal)


In [ ]:
plot_staircase(
    cal["window_df"], cal["steps"], pressure_col="pressure_psi_barocorr",
    out_path=os.path.join(site_dir, "staircase.png"),
)
plt.show()

plot_calibration_fit(cal["fit_result"], out_path=os.path.join(site_dir, "calibration_fit.png"))
plt.show()


## 6. Run the deployment (wave file)

`process_deployment()` applies the calibration fit from section 5 to
`SITE["raw_csv"]`, trimmed to `[survey_start, survey_end]`, and writes
`clean.csv`/`oceanlyz.csv` into `site_dir`. If `recovery_depth_m` was set,
it also prints a rough cross-check against the modeled depth at the end of
the record.

In [ ]:
clean_df, report, raw_df = process_deployment(SITE, cal, input_dir=INPUT_DIR, output_dir=site_dir)
print_report(report)

summary_path = os.path.join(OUTPUT_DIR, "processing_summary.csv")
summary_row = {k: v for k, v in report.items() if k != "recovery_check"}
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    summary = summary[summary["site"] != report["site"]]  # drop any previous row for this site
    summary = pd.concat([summary, pd.DataFrame([summary_row])], ignore_index=True)
else:
    summary = pd.DataFrame([summary_row])
summary.to_csv(summary_path, index=False)
summary


In [ ]:
plot_qc(raw_df, clean_df, SITE["name"], out_path=os.path.join(site_dir, "qc.png"))
plt.show()


## Next steps

- Another sensor: edit `CALIB` and `SITE` in section 4 and rerun from there.
- Feed `<name>/oceanlyz.csv` into OCEANLYZ (`InputType='pressure'`, `fs`
  and `heightfrombed` are printed above) for the actual wave analysis.